# 30 — Surgical Cross-Model Survey Replay

**Question:** Is the Day-1 plateau and the persistent ~+0.4 prompt-chain drift observed in R14 split50 (Qwen3-8B) a property of *Qwen* or a property of *the simulation paradigm*?

**Approach (Option C — multi-day frozen-reflection replay):**
- Take 20 stratified agents (10 A-only + 10 B-only) from R14 split50.
- **Freeze** every upstream state Qwen produced: Day-0 anchor, Day-0 rationales, broadcast reflections, daily summaries.
- For each (agent, day∈{1,2,3}, policy∈6), give each candidate model the **exact same** `assemble_context()` system prompt Qwen saw.
- Each model runs the 2-step debias chain and returns its own reasoning + A-G letter + numeric.
- Compare to Qwen's saved `opinion_trajectories.csv` cell-by-cell.

**Models** (override below if API model strings change):
- `claude-sonnet-4-6` (anthropic)
- `claude-haiku-4-5-20251001` (anthropic)
- `gpt-5.4-mini` (openai)

**Budget:** 3 models × 20 agents × 3 days × 6 policies × 2 debias steps = **2,160 calls** (~$8–12).

**Source artefacts (frozen):** `data/output/experiments/run_6202348_R14_split50_5day/20260619_192129/`

**Output:** `data/output/calibration/30_replay_<timestamp>/`

**No agent state is mutated.** The replay helper builds the prompt, calls `send_chat`, returns the result — `administer_survey()` is *not* called.

---

## How to use

1. **Cells 1–6** are setup; safe to run unconditionally.
2. **Cell 7** is a single-call smoke (1 LLM call) to confirm wiring before spending real budget.
3. **Cell 9** is the full matrix execution. It is **idempotent** — if a results CSV already exists in the output dir, completed cells are skipped. Set `DRY_RUN = True` in Cell 8 to verify the loop wiring with only 1 agent × 1 day × 1 policy per model (6 calls total).
4. **Cells 10–12** are analysis.

In [1]:
# Cell 1 — Imports + paths
from __future__ import annotations

import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve repo root robustly whether the kernel is launched from /notebooks
# or the repo root.
_here = Path.cwd()
REPO = _here if (_here / "src" / "cag").exists() else _here.parent
assert (REPO / "src" / "cag").exists(), f"Could not find src/cag from {_here}"
sys.path.insert(0, str(REPO / "src"))

from cag.abm.agent import (
    SurveyedCitizen,
    _ANTI_SYCOPHANCY,
    _DEBIAS_STEP1_TEMPLATE,
    _DEBIAS_STEP2_TEMPLATE,
)
from cag.abm.attributes.opinion import (
    ALL_CLIMATE_POLICIES,
    ClimatePolicyID,
    PACKAGE_SCOPE,
    RESPONSE_LABELS,
    RESPONSE_SCALE,
    SURVEY_QUESTIONS,
)
from cag.io.llm import load_api_key, parse_letter_response, send_chat
from cag.io.survey import load

# Frozen source
SRC_RUN = REPO / "data" / "output" / "experiments" / "run_6202348_R14_split50_5day" / "20260619_192129"
assert SRC_RUN.exists(), f"Missing R14 split50 source dir: {SRC_RUN}"

# Output
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = REPO / "data" / "output" / "calibration" / f"30_replay_{STAMP}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"REPO    : {REPO}")
print(f"SRC_RUN : {SRC_RUN}")
print(f"OUT_DIR : {OUT_DIR}")

REPO    : /Users/vbwt265/src/GitHub/Climate-Action-GABM
SRC_RUN : /Users/vbwt265/src/GitHub/Climate-Action-GABM/data/output/experiments/run_6202348_R14_split50_5day/20260619_192129
OUT_DIR : /Users/vbwt265/src/GitHub/Climate-Action-GABM/data/output/calibration/30_replay_20260620_162106


In [2]:
# Cell 2 — Load Qwen artefacts
REF_CSV   = SRC_RUN / "reflections.csv"
SUM_CSV   = SRC_RUN / "daily_summaries.csv"
SR_CSV    = SRC_RUN / "survey_reasoning.csv"
OT_CSV    = SRC_RUN / "opinion_trajectories.csv"
MSG_CSV   = SRC_RUN / "messages.csv"
GT_CSV    = SRC_RUN / "ground_truth.csv"
CFG_JSON  = SRC_RUN / "config.json"

qwen_ref  = pd.read_csv(REF_CSV)
qwen_sum  = pd.read_csv(SUM_CSV)
qwen_sr   = pd.read_csv(SR_CSV)
qwen_ot   = pd.read_csv(OT_CSV)
qwen_msg  = pd.read_csv(MSG_CSV)
qwen_gt   = pd.read_csv(GT_CSV)
qwen_cfg  = json.loads(CFG_JSON.read_text())

print(f"reflections        : {len(qwen_ref):>5}  agents={qwen_ref['agent_id'].nunique()}  days={sorted(qwen_ref['day'].unique().tolist())}")
print(f"daily_summaries    : {len(qwen_sum):>5}  agents={qwen_sum['agent_id'].nunique()}  days={sorted(qwen_sum['day'].unique().tolist())}")
print(f"survey_reasoning   : {len(qwen_sr):>5}  agents={qwen_sr['agent_id'].nunique()}  days={sorted(qwen_sr['day'].unique().tolist())}")
print(f"opinion_trajectory : {len(qwen_ot):>5}  agents={qwen_ot['agent_id'].nunique()}  days={sorted(qwen_ot['day'].unique().tolist())}")
print(f"messages           : {len(qwen_msg):>5}")
print(f"ground_truth       : {len(qwen_gt):>5}")
print(f"config provider    : {qwen_cfg.get('llm_provider')!r}, model {qwen_cfg.get('llm_model')!r}")
print(f"comm_mode          : {qwen_cfg.get('communication_mode')!r}")

reflections        :   250  agents=50  days=[1, 2, 3, 4, 5]
daily_summaries    :   150  agents=50  days=[1, 2, 3]
survey_reasoning   :  1800  agents=50  days=[0, 1, 2, 3, 4, 5]
opinion_trajectory :  1800  agents=50  days=[0, 1, 2, 3, 4, 5]
messages           :   250
ground_truth       :   300
config provider    : 'local', model 'Qwen/Qwen3-8B'
comm_mode          : 'package'


In [3]:
# Cell 3 — Rebuild nation, infer buckets, stratified sample of 20 agents
# (10 A-only + 10 B-only).  Bucket is inferred from each agent's reflections
# 'phase' column: in split50 each agent only ever sees ONE side of broadcast,
# so they only write reflections during ONE of the P-A / P-B phases.

from cag.__main__ import build_nation

RANDOM_SEED_NATION  = qwen_cfg.get("random_seed", 42)
RANDOM_SEED_SUBSET  = 123  # independent of the run seed for the 20-pick
N_PER_BUCKET        = 10

YOUGOV_CSV = REPO / "data" / "yougov_survey_data" / "YouGovProcessedData.csv"
assert YOUGOV_CSV.exists(), f"Missing YouGov CSV: {YOUGOV_CSV}"

data_full = load(str(YOUGOV_CSV))
data50    = data_full.sample(n=qwen_cfg["n_citizens"], random_state=RANDOM_SEED_NATION).reset_index(drop=True)
nation50  = build_nation(data50)
print(f"Rebuilt nation with {len(nation50.agents_active)} agents.")

# Bucket inference from broadcast reflections.
broadcast_ref = qwen_ref[qwen_ref["phase"].isin(["P-A", "P-B"])]
phase_per_agent = broadcast_ref.groupby("agent_id")["phase"].first()
bucket_map = phase_per_agent.map({"P-A": "A-only", "P-B": "B-only"})
print("\nBucket distribution (full 50):")
print(bucket_map.value_counts())

rng = np.random.default_rng(RANDOM_SEED_SUBSET)
a_pool = bucket_map[bucket_map == "A-only"].index.tolist()
b_pool = bucket_map[bucket_map == "B-only"].index.tolist()
a_pick = list(rng.choice(a_pool, size=N_PER_BUCKET, replace=False))
b_pick = list(rng.choice(b_pool, size=N_PER_BUCKET, replace=False))
PICKED = sorted(a_pick + b_pick)
PICKED_BUCKET = {aid: ("A-only" if aid in a_pick else "B-only") for aid in PICKED}
print(f"\nPicked {len(PICKED)} agents ({len(a_pick)} A-only + {len(b_pick)} B-only).")
print(f"Agent IDs (first 5): {PICKED[:5]}...")

Rebuilt nation with 50 agents.

Bucket distribution (full 50):
phase
B-only    25
A-only    25
Name: count, dtype: int64

Picked 20 agents (10 A-only + 10 B-only).
Agent IDs (first 5): [np.float64(69.0), np.float64(91.0), np.float64(165.0), np.float64(166.0), np.float64(238.0)]...


In [4]:
# Cell 4 — inject_qwen_state(): hydrate the SurveyedCitizen with Qwen's state.
#
# After this runs, agent.assemble_context(day, policy_id) for day∈{1,2,3} will
# return the exact same string Qwen saw at survey time. assemble_context already
# filters by day at runtime (summaries with d < day-1; reflections in {day-1, day})
# so we simply over-inject the full state and let the existing filter do the slicing.

_PID_FROM_STR = {str(pid): pid for pid in ALL_CLIMATE_POLICIES}
_PID_FROM_STR[PACKAGE_SCOPE] = PACKAGE_SCOPE

def _to_policy(value):
    if value is None or value == "" or (isinstance(value, float) and pd.isna(value)):
        return ""
    return _PID_FROM_STR.get(str(value), str(value))

def inject_qwen_state(agent, agent_id, ref_df, sum_df, sr_df, ot_df):
    """Populate agent.{reflections, daily_summaries, survey_reasoning, opinion_history}
    from the Qwen CSV slices for this agent_id.  No LLM calls."""
    aid = agent_id

    # 1. reflections (full list; assemble_context filters by day)
    agent.reflections = []
    for row in ref_df[ref_df["agent_id"] == aid].itertuples(index=False):
        try:
            messages_received = json.loads(row.messages_received_json)
        except (TypeError, ValueError):
            messages_received = []
        try:
            policy_ids = json.loads(row.policy_ids_json)
        except (TypeError, ValueError):
            policy_ids = []
        entry = {
            "day": int(row.day),
            "phase": row.phase,
            "policy_id": _to_policy(row.policy_id),
            "text": row.text,
            "messages_received": messages_received,
        }
        if policy_ids:
            entry["policy_ids"] = [_to_policy(p) for p in policy_ids]
        agent.reflections.append(entry)

    # 2. daily_summaries keyed (day, policy_id)
    agent.daily_summaries = {}
    for row in sum_df[sum_df["agent_id"] == aid].itertuples(index=False):
        agent.daily_summaries[(int(row.day), _to_policy(row.policy_id))] = row.summary

    # 3. survey_reasoning keyed policy_id -> [(day, text), ...]
    agent.survey_reasoning = {}
    for row in sr_df[sr_df["agent_id"] == aid].itertuples(index=False):
        pid = _to_policy(row.policy_id)
        agent.survey_reasoning.setdefault(pid, []).append((int(row.day), row.reasoning))

    # 4. opinion_history keyed policy_id -> [(day, numeric), ...]
    agent.opinion_history = {}
    for row in ot_df[ot_df["agent_id"] == aid].itertuples(index=False):
        pid = _to_policy(row.policy_id)
        agent.opinion_history.setdefault(pid, []).append((int(row.day), int(row.numeric)))

    return agent

# Apply to the 20 picked agents in-place.
for aid in PICKED:
    inject_qwen_state(
        nation50.agents_active[aid], aid,
        qwen_ref, qwen_sum, qwen_sr, qwen_ot,
    )
print(f"Hydrated {len(PICKED)} agents from Qwen state.")

# Quick state spot-check.
sample = nation50.agents_active[PICKED[0]]
print(f"\nSample agent {PICKED[0]} ({PICKED_BUCKET[PICKED[0]]}):")
print(f"  reflections        : {len(sample.reflections):>3}  (days {sorted({r['day'] for r in sample.reflections})})")
print(f"  daily_summaries    : {len(sample.daily_summaries):>3}  keys {sorted(sample.daily_summaries.keys())}")
print(f"  survey_reasoning   : {len(sample.survey_reasoning):>3} policies")
print(f"  opinion_history    : {len(sample.opinion_history):>3} policies")

Hydrated 20 agents from Qwen state.

Sample agent 69.0 (B-only):
  reflections        :   5  (days [1, 2, 3, 4, 5])
  daily_summaries    :   3  keys [(1, 'climate_policy_package'), (2, 'climate_policy_package'), (3, 'climate_policy_package')]
  survey_reasoning   :   6 policies
  opinion_history    :   6 policies


In [5]:
# Cell 5 — Substring sanity check: verify each agent's reconstructed Day-1
# and Day-3 prompt contains Qwen's saved Day-0 rationale.
#
# Note: in package mode, reflections are stored under policy_id=PACKAGE_SCOPE,
# while administer_survey() calls assemble_context() with policy_id=<single
# policy>. assemble_context() filters reflections by exact policy_id match, so
# package reflections do NOT appear in per-policy survey contexts. This is the
# production behavior we are faithfully replaying. We separately verify that
# the package reflection appears in a package-scoped context as a diagnostic.

def _first_words(text, n=20):
    return " ".join(str(text).split()[:n])

PROBE_POLICY = ClimatePolicyID.RENEWABLE_ENERGY
checked, ok = 0, 0
for aid in PICKED[:5]:
    agent = nation50.agents_active[aid]
    ctx1 = agent.assemble_context(day=1, policy_id=PROBE_POLICY)
    ctx3 = agent.assemble_context(day=3, policy_id=PROBE_POLICY)
    ctx3_pkg = agent.assemble_context(day=3, policy_id=PACKAGE_SCOPE)

    # Day-0 rationale should appear in BOTH Day-1 and Day-3 per-policy contexts.
    day0_text = None
    for d, t in agent.survey_reasoning.get(PROBE_POLICY, []):
        if d == 0:
            day0_text = t
            break
    assert day0_text is not None, f"agent {aid} has no Day-0 rationale for {PROBE_POLICY}"
    snippet = _first_words(day0_text, 8)
    in1 = snippet in ctx1
    in3 = snippet in ctx3

    # Day-2 package reflection should appear in a Day-3 package-scoped context.
    day2_ref = [r for r in agent.reflections if r["day"] == 2]
    ref_snippet = _first_words(day2_ref[0]["text"], 8) if day2_ref else ""
    ref_in3_pkg = ref_snippet in ctx3_pkg if ref_snippet else True

    checked += 1
    if in1 and in3 and ref_in3_pkg:
        ok += 1
    else:
        print(f"  agent {aid}: day0_in_ctx1={in1}, day0_in_ctx3={in3}, day2ref_in_ctx3_pkg={ref_in3_pkg}")
        print(f"    day0 snippet     : {snippet!r}")
        print(f"    day2ref snippet  : {ref_snippet!r}")

print(f"\nSubstring sanity: {ok}/{checked} agents passed all three checks.")
assert ok == checked, "State injection sanity check failed; aborting before LLM spend."



Substring sanity: 5/5 agents passed all three checks.


In [6]:
# Cell 6 — replay_survey(): the 2-step debias chain WITHOUT mutating the agent.
# Mirrors administer_survey() in src/cag/abm/agent.py but returns instead of writing.

def replay_survey(agent, day, policy_id, model, provider, api_key=None,
                  temperature=0.5, thinking=False):
    """Run the debias 2-step survey on a frozen agent. No state writes.

    Returns a dict with keys:
      reasoning, raw_response, letter, numeric, latency_s, error
    On any exception, `error` is the string message and the other fields
    are None; the caller decides whether to retry or skip.
    """
    out = {
        "reasoning": None, "raw_response": None, "letter": None,
        "numeric": None, "latency_s": None, "error": None,
    }
    t0 = time.perf_counter()
    try:
        system_prompt = agent.get_system_prompt(day=day, policy_id=policy_id)
        policy_question = SURVEY_QUESTIONS[policy_id]
        step1_prompt = _DEBIAS_STEP1_TEMPLATE.format(
            anti_sycophancy=_ANTI_SYCOPHANCY,
            policy_question=policy_question,
        )
        reasoning = send_chat(
            system_prompt, step1_prompt, api_key=api_key, model=model,
            provider=provider, temperature=temperature, thinking=thinking,
        )
        response_options = "\n".join(
            f"{letter}. {label}" for letter, label in RESPONSE_LABELS.items()
        )
        step2_system = system_prompt + "\n\nYour reasoning about this policy:\n" + reasoning
        step2_prompt = _DEBIAS_STEP2_TEMPLATE.format(
            policy_question=policy_question,
            response_options=response_options,
        )
        raw = send_chat(
            step2_system, step2_prompt, api_key=api_key, model=model,
            provider=provider, temperature=temperature, thinking=thinking,
        )
        letter = parse_letter_response(raw)
        numeric = RESPONSE_SCALE.get(letter)
        out.update(reasoning=reasoning, raw_response=raw, letter=letter, numeric=numeric)
    except Exception as exc:
        out["error"] = f"{type(exc).__name__}: {exc}"
    finally:
        out["latency_s"] = time.perf_counter() - t0
    return out

In [7]:
# Cell 7 — Single-call smoke. One LLM call against the cheapest available
# provider to confirm wiring before the full matrix run.

SMOKE_AGENT  = PICKED[0]
SMOKE_DAY    = 1
SMOKE_POLICY = ClimatePolicyID.RENEWABLE_ENERGY

# Pick a provider that has a key on disk. Order: openai → anthropic → local.
_smoke_choice = None
for prov, mdl in [("openai", "gpt-5.4-mini"), ("anthropic", "claude-haiku-4-5-20251001")]:
    try:
        k = load_api_key(prov)
        if k:
            _smoke_choice = (prov, mdl, k)
            break
    except Exception:
        continue

if _smoke_choice is None:
    print("WARN: no API key found on disk — skipping single-call smoke.")
else:
    prov, mdl, key = _smoke_choice
    print(f"Smoke test: provider={prov!r} model={mdl!r} agent={SMOKE_AGENT} day={SMOKE_DAY} policy={SMOKE_POLICY!r}")
    res = replay_survey(
        nation50.agents_active[SMOKE_AGENT], SMOKE_DAY, SMOKE_POLICY,
        model=mdl, provider=prov, api_key=key,
    )
    if res["error"]:
        print(f"  ERROR: {res['error']}")
    else:
        print(f"  letter={res['letter']!r}  numeric={res['numeric']}  latency={res['latency_s']:.2f}s")
        print(f"  reasoning (first 240 chars): {(res['reasoning'] or '')[:240]}")
        print(f"  raw_response (first 120 chars): {(res['raw_response'] or '')[:120]}")

Smoke test: provider='openai' model='gpt-5.4-mini' agent=69.0 day=1 policy=ClimatePolicyID(1)


  letter='E'  numeric=1  latency=2.62s
  reasoning (first 240 chars): I would slightly support accelerating renewable energy rollout. My concern for the environment, climate responsibility, and long-term sustainability makes me favour more wind and other renewables, especially as someone who voted Remain and 
  raw_response (first 120 chars): E


In [8]:
# Cell 8 — Model registry + run-mode switches.
#
# Set DRY_RUN=True to verify the loop wiring with 1 agent × 1 day × 1 policy
# per model (= 3 calls × 2 steps = 6 LLM calls total). DRY_RUN=False runs the
# full 2160-call matrix.

DRY_RUN = False   # ← flip to False for the full run

MODEL_REGISTRY = [
    {"label": "claude-sonnet-4-6",     "provider": "anthropic", "model": "claude-sonnet-4-6"},
    {"label": "claude-haiku-4-5",      "provider": "anthropic", "model": "claude-haiku-4-5-20251001"},
    {"label": "gpt-5.4-mini",          "provider": "openai",    "model": "gpt-5.4-mini"},
]

# Days and policies in scope.
DAYS     = [1, 2, 3]
POLICIES = list(ALL_CLIMATE_POLICIES)   # all 6

# Resolve API keys once; skip models whose key is missing.
ACTIVE_MODELS = []
for entry in MODEL_REGISTRY:
    try:
        key = load_api_key(entry["provider"])
        if key:
            ACTIVE_MODELS.append({**entry, "api_key": key})
        else:
            print(f"  skip {entry['label']!r}: empty key for provider {entry['provider']!r}")
    except Exception as exc:
        print(f"  skip {entry['label']!r}: no key ({exc})")

if DRY_RUN:
    AGENTS_RUN   = PICKED[:1]
    DAYS_RUN     = DAYS[:1]
    POLICIES_RUN = POLICIES[:1]
else:
    AGENTS_RUN   = PICKED
    DAYS_RUN     = DAYS
    POLICIES_RUN = POLICIES

expected_calls = len(ACTIVE_MODELS) * len(AGENTS_RUN) * len(DAYS_RUN) * len(POLICIES_RUN) * 2
print(f"\nDRY_RUN={DRY_RUN}")
print(f"Active models : {[m['label'] for m in ACTIVE_MODELS]}")
print(f"Agents        : {len(AGENTS_RUN)}")
print(f"Days          : {DAYS_RUN}")
print(f"Policies      : {len(POLICIES_RUN)}")
print(f"Expected LLM calls (2 per cell): {expected_calls}")


DRY_RUN=False
Active models : ['claude-sonnet-4-6', 'claude-haiku-4-5', 'gpt-5.4-mini']
Agents        : 20
Days          : [1, 2, 3]
Policies      : 6
Expected LLM calls (2 per cell): 2160


In [9]:
# Cell 9 — Replay matrix execution.
#
# Idempotent: if RESULTS_CSV already contains a (model_label, agent_id, day, policy_id)
# tuple, we skip it. After each model finishes, we write the cumulative CSV atomically.
# A kill mid-run leaves the most recent model-fully-completed state on disk.

RESULTS_CSV = OUT_DIR / "results.csv"
RESULTS_COLS = [
    "model_label", "provider", "model", "agent_id", "bucket",
    "day", "policy_id", "reasoning", "raw_response",
    "letter", "numeric", "latency_s", "error",
]

if RESULTS_CSV.exists():
    existing = pd.read_csv(RESULTS_CSV)
    done_keys = set(
        zip(existing["model_label"], existing["agent_id"], existing["day"], existing["policy_id"])
    )
    rows = existing.to_dict("records")
    print(f"Resuming: {len(rows)} rows already on disk; will skip duplicates.")
else:
    done_keys = set()
    rows = []

def _atomic_dump():
    df = pd.DataFrame(rows, columns=RESULTS_COLS)
    tmp = RESULTS_CSV.with_suffix(".csv.tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(RESULTS_CSV)

overall_t0 = time.perf_counter()
for m in ACTIVE_MODELS:
    label, prov, mdl, key = m["label"], m["provider"], m["model"], m["api_key"]
    print(f"\n=== {label}  ({prov} / {mdl}) ===")
    model_t0 = time.perf_counter()
    n_done, n_skip, n_err = 0, 0, 0
    for aid in AGENTS_RUN:
        agent = nation50.agents_active[aid]
        bucket = PICKED_BUCKET[aid]
        for day in DAYS_RUN:
            for policy in POLICIES_RUN:
                key_tuple = (label, aid, day, str(policy))
                if key_tuple in done_keys:
                    n_skip += 1
                    continue
                res = replay_survey(agent, day, policy, model=mdl, provider=prov, api_key=key)
                rows.append({
                    "model_label": label, "provider": prov, "model": mdl,
                    "agent_id": aid, "bucket": bucket,
                    "day": day, "policy_id": str(policy),
                    "reasoning": res["reasoning"], "raw_response": res["raw_response"],
                    "letter": res["letter"], "numeric": res["numeric"],
                    "latency_s": res["latency_s"], "error": res["error"],
                })
                done_keys.add(key_tuple)
                n_done += 1
                if res["error"]:
                    n_err += 1
    _atomic_dump()
    elapsed = time.perf_counter() - model_t0
    print(f"  {label}: {n_done} done, {n_skip} skipped, {n_err} errors  [{elapsed:.1f}s]")

total_elapsed = time.perf_counter() - overall_t0
print(f"\nAll models complete. {len(rows)} total rows, {total_elapsed:.1f}s. Wrote {RESULTS_CSV}.")


=== claude-sonnet-4-6  (anthropic / claude-sonnet-4-6) ===
  claude-sonnet-4-6: 360 done, 0 skipped, 0 errors  [2198.9s]

=== claude-haiku-4-5  (anthropic / claude-haiku-4-5-20251001) ===
  claude-haiku-4-5: 360 done, 0 skipped, 0 errors  [1588.6s]

=== gpt-5.4-mini  (openai / gpt-5.4-mini) ===
  gpt-5.4-mini: 360 done, 0 skipped, 0 errors  [923.5s]

All models complete. 1080 total rows, 4711.0s. Wrote /Users/vbwt265/src/GitHub/Climate-Action-GABM/data/output/calibration/30_replay_20260620_162106/results.csv.


In [10]:
# Cell 10 — Analysis 1: per-model bucket trajectory.
# Headline plot: mean numeric per (day, bucket) per model, with Qwen's saved
# trajectory for the same 20 agents overlaid as a reference dotted line.

import matplotlib.pyplot as plt

res_df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(columns=RESULTS_COLS)
if res_df.empty:
    print("No replay results on disk yet — run Cell 9.")
else:
    # Replay means.
    replay_clean = res_df.dropna(subset=["numeric"]).copy()
    replay_means = (replay_clean.groupby(["model_label", "day", "bucket"])["numeric"]
                                 .mean().reset_index())

    # Qwen reference for the SAME 20 agents on the SAME days.
    qwen_subset = qwen_ot[
        qwen_ot["agent_id"].isin(PICKED) &
        qwen_ot["day"].isin([0] + list(DAYS))
    ].copy()
    qwen_subset["bucket"] = qwen_subset["agent_id"].map(PICKED_BUCKET)
    qwen_means = qwen_subset.groupby(["day", "bucket"])["numeric"].mean().reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
    for ax, bucket in zip(axes, ["A-only", "B-only"]):
        for label, sub in replay_means[replay_means["bucket"] == bucket].groupby("model_label"):
            sub = sub.sort_values("day")
            ax.plot(sub["day"], sub["numeric"], marker="o", label=label)
        qw = qwen_means[qwen_means["bucket"] == bucket].sort_values("day")
        ax.plot(qw["day"], qw["numeric"], marker="x", linestyle=":", color="black", label="qwen3-8b (saved)")
        ax.set_title(f"{bucket}")
        ax.set_xlabel("Day")
        ax.axhline(0, color="grey", linewidth=0.5)
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("Mean numeric opinion (–3..+3)")
    axes[1].legend(loc="best", fontsize=8)
    fig.suptitle(f"NB 30 — Bucket-mean opinion across models (n={len(PICKED)//2} per bucket, all 6 policies pooled)")
    fig.tight_layout()
    plt.savefig(OUT_DIR / "bucket_trajectory.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved {OUT_DIR / 'bucket_trajectory.png'}")

Saved /Users/vbwt265/src/GitHub/Climate-Action-GABM/data/output/calibration/30_replay_20260620_162106/bucket_trajectory.png


/var/folders/9r/wv909y5d5tz8vdmlx6gll_2h0000gr/T/ipykernel_68057/230382778.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# Cell 11 — Analysis 2: per-cell divergence from Qwen.
# For every (agent, day, policy), compute |model_numeric − qwen_saved_numeric|.

if res_df.empty:
    print("No replay results on disk yet — run Cell 9.")
else:
    # Join replay rows to Qwen saved numeric on (agent_id, day, policy_id).
    qwen_lookup = qwen_ot.copy()
    qwen_lookup["policy_id"] = qwen_lookup["policy_id"].astype(str)
    qwen_lookup = qwen_lookup.rename(columns={"numeric": "qwen_numeric"})
    joined = replay_clean.merge(
        qwen_lookup[["agent_id", "day", "policy_id", "qwen_numeric"]],
        on=["agent_id", "day", "policy_id"], how="left",
    )
    joined["abs_diff"] = (joined["numeric"] - joined["qwen_numeric"]).abs()

    summary = (joined.groupby("model_label")["abs_diff"]
                     .agg(n="size", median="median", p90=lambda s: s.quantile(0.9),
                          ge1=lambda s: (s >= 1).mean(),
                          ge2=lambda s: (s >= 2).mean(),
                          mean="mean")
                     .round(3))
    print("Per-cell |replay − qwen_saved|:")
    print(summary.to_string())
    summary.to_csv(OUT_DIR / "divergence_summary.csv")
    print(f"\nSaved {OUT_DIR / 'divergence_summary.csv'}")

Per-cell |replay − qwen_saved|:
                     n  median  p90    ge1    ge2   mean
model_label                                             
claude-haiku-4-5   360     0.0  2.0  0.356  0.142  0.586
claude-sonnet-4-6  360     0.0  1.0  0.278  0.081  0.378
gpt-5.4-mini       360     0.0  1.0  0.197  0.050  0.267

Saved /Users/vbwt265/src/GitHub/Climate-Action-GABM/data/output/calibration/30_replay_20260620_162106/divergence_summary.csv


In [12]:
# Cell 12 — Analysis 3: stratified rationale prose comparison.
# Pick a small panel of (agent, bucket) cells and print all models' rationale
# side-by-side for Day 3 Carbon Tax. User reads → judges richer-reasoning.

PROSE_DAY    = 3
PROSE_POLICY = ClimatePolicyID.CARBON_TAX

if res_df.empty:
    print("No replay results on disk yet — run Cell 9.")
else:
    # Pick 2 agents per bucket (first 2 by id for reproducibility).
    pick_a = sorted([a for a in PICKED if PICKED_BUCKET[a] == "A-only"])[:2]
    pick_b = sorted([a for a in PICKED if PICKED_BUCKET[a] == "B-only"])[:2]
    prose_agents = pick_a + pick_b

    for aid in prose_agents:
        bucket = PICKED_BUCKET[aid]
        print("=" * 88)
        print(f"AGENT {aid}  ({bucket})  —  Day {PROSE_DAY}  —  {PROSE_POLICY!r}")
        print("=" * 88)
        # Qwen's saved Day-3 reasoning for context.
        qwen_row = qwen_sr[
            (qwen_sr["agent_id"] == aid) &
            (qwen_sr["day"] == PROSE_DAY) &
            (qwen_sr["policy_id"] == str(PROSE_POLICY))
        ]
        qwen_num_row = qwen_ot[
            (qwen_ot["agent_id"] == aid) &
            (qwen_ot["day"] == PROSE_DAY) &
            (qwen_ot["policy_id"] == str(PROSE_POLICY))
        ]
        qwen_num = qwen_num_row["numeric"].iloc[0] if not qwen_num_row.empty else None
        print(f"\n[qwen3-8b (saved)] numeric={qwen_num}")
        if not qwen_row.empty:
            print(qwen_row["reasoning"].iloc[0])
        else:
            print("(no Qwen reasoning row found)")

        for label in res_df["model_label"].unique():
            sub = res_df[
                (res_df["model_label"] == label) &
                (res_df["agent_id"] == aid) &
                (res_df["day"] == PROSE_DAY) &
                (res_df["policy_id"] == str(PROSE_POLICY))
            ]
            if sub.empty:
                continue
            row = sub.iloc[0]
            print(f"\n[{label}] numeric={row['numeric']}  letter={row['letter']!r}")
            print(row["reasoning"] or "(no reasoning)")
        print()

AGENT 91.0  (A-only)  —  Day 3  —  ClimatePolicyID(5)

[qwen3-8b (saved)] numeric=2
I would support a carbon tax with revenue distributed to the public because it addresses environmental concerns without disproportionately burdening low-income households, and I believe in fair treatment of all citizens. However, I am cautious about rapid change and prefer stable, traditional approaches, so I would want the policy to be carefully implemented and not disrupt existing industries or ways of life.

[claude-sonnet-4-6] numeric=2  letter='F'
As someone who voted Remain and Labour, I'm broadly sympathetic to policies that address environmental concerns while also redistributing benefits fairly to ordinary people, which aligns with my belief in equal opportunities and fair treatment. However, as someone slightly right-of-centre who values stability and traditional approaches, I'd want reassurance that this wouldn't unduly disrupt established industries or push up energy costs for households bef